# Additive Symbolic Regression

Additive symbolic regression fits a model as a **sum of small symbolic expressions**,
$f(x) = c + \sum_k \eta_k\, g_k(x)$, where each $g_k$ is discovered by the existing JAXSR machinery.
This is analogous to **gradient boosting**, except each weak learner is an interpretable symbolic expression rather than a decision tree.

This notebook shows the `StagewiseSymbolicRegressor`: it repeatedly fits a small expression to the current residual, then (optionally) refits all linear coefficients by least squares.

In [1]:
import numpy as np

from jaxsr.additive import StagewiseSymbolicRegressor

# Additive target: y = 2*x0 + 0.5*x1^2 + noise
rng = np.random.default_rng(0)
X = rng.uniform(-2, 2, size=(300, 2))
y = 2.0 * X[:, 0] + 0.5 * X[:, 1] ** 2 + 0.1 * rng.normal(size=300)

# hold out a test set
n_train = 200
X_train, y_train = X[:n_train], y[:n_train]
X_test, y_test = X[n_train:], y[n_train:]
X_train.shape, X_test.shape

((200, 2), (100, 2))

## Fit a stagewise additive model

Keep `max_complexity` small so each stage is a simple expression and the ensemble accumulates many interpretable terms. With `refit_coefficients=True`, all linear weights are re-solved by least squares after each stage.

In [2]:
model = StagewiseSymbolicRegressor(
    n_terms=5,
    learning_rate=0.2,
    max_complexity=4,
    refit_coefficients=True,
)
model.fit(X_train, y_train)
print(model)

StagewiseSymbolicRegressor(
    intercept = 1.075
    terms =
        + 1.001 * (y = 1.998*x0 - 1.07 + 0.5015*x1^2)
        + 2.31 * (y = - 0.003903*x0^2)
        + 1.865 * (y = 0.002939*x1)
        + 1.124 * (y = 0.001106*x0*x1)
        + 6.931 * (y = - 0.0001612*x1^3)
)


## Inspect the learned ensemble

In [3]:
print('intercept:', model.intercept_)
print('coefficients:', model.coefficients_)
print()
for i, expr in enumerate(model.expressions_):
    print(f'term {i}: {expr}')

intercept: 1.0750973224639893
coefficients: [1.0005701780319214, 2.3100991249084473, 1.8648916482925415, 1.1235284805297852, 6.930931568145752]

term 0: y = 1.998*x0 - 1.07 + 0.5015*x1^2
term 1: y = - 0.003903*x0^2
term 2: y = 0.002939*x1
term 3: y = 0.001106*x0*x1
term 4: y = - 0.0001612*x1^3


In [4]:
print('train R^2:', model.score(X_train, y_train))
print('test  R^2:', model.score(X_test, y_test))

train R^2: 0.9982670545578003


test  R^2: 0.9988469481468201


## Combined symbolic expression

`to_expression()` collapses the ensemble into a single simplified SymPy expression.

In [5]:
model.to_expression()

0.00124225317639448*x0*x1 + 1.99923764521122*x0 - 0.0090172041649631*x0**2.0 + 0.00548121186382547*x1 + 0.501793767439942*x1**2.0 - 0.00111730957318264*x1**3.0 + 0.00453644099624739

## Why additive? Many small terms vs one big expression

When the signal is a sum of several simple effects, a tiny per-stage budget still recovers it because the ensemble accumulates terms across stages — where a single expression with the same small budget cannot.

In [6]:
from jaxsr import fit_symbolic

rng = np.random.default_rng(2)
Xm = rng.uniform(-1.5, 1.5, size=(500, 4))
ym = (
    1.5 * Xm[:, 0]
    + 2.0 * Xm[:, 1] ** 2
    - 1.0 * Xm[:, 2] ** 3
    + 0.8 * Xm[:, 3]
    + 0.2 * rng.normal(size=500)
)
Xtr, ytr, Xte, yte = Xm[:300], ym[:300], Xm[300:], ym[300:]

stagewise = StagewiseSymbolicRegressor(n_terms=10, max_complexity=1).fit(Xtr, ytr)
single = fit_symbolic(Xtr, ytr, max_terms=1, include_transcendental=False)

print('stagewise (max_complexity=1, 10 stages) test R^2:', round(float(stagewise.score(Xte, yte)), 4))
print('single expression (max_terms=1)         test R^2:', round(float(single.score(Xte, yte)), 4))

stagewise (max_complexity=1, 10 stages) test R^2: 0.9931
single expression (max_terms=1)         test R^2: 0.3119


## Early stopping and saving

Early stopping holds out a validation split and stops once it stops improving. Models serialize to JSON (they are not picklable because the basis functions use closures).

In [7]:
import tempfile, os

es = StagewiseSymbolicRegressor(
    n_terms=20, max_complexity=3,
    early_stopping=True, validation_fraction=0.25, patience=3, random_state=0,
).fit(X_train, y_train)
print('terms used after early stopping:', es.n_terms_)

path = os.path.join(tempfile.mkdtemp(), 'additive_model.json')
es.save(path)
loaded = StagewiseSymbolicRegressor.load(path)
print('reloaded test R^2:', round(float(loaded.score(X_test, y_test)), 4))
os.remove(path)

terms used after early stopping: 2
reloaded test R^2: 0.9989


## Difference from single-expression and backfitting

- **Single-expression** (`jaxsr.SymbolicRegressor`): one sparse expression over a fixed basis library.
- **Stagewise additive** (this notebook): terms are discovered one at a time on the residual and then **frozen**.
- **Backfitting additive** (`BackfittingSymbolicRegressor`, planned): terms are **revised** repeatedly, each conditioned on the others (BART/iBART-style).

See `docs/guides/additive-symbolic-regression.md` for the full guide.